# Day 11 – Fine‑tuning with LoRA

In [ ]:
# Install required packages
# !pip install transformers datasets peft accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType

# Load base model and tokenizer
model_name = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Create a tiny dataset (Q&A pairs)
data = [
    {"instruction": "What is the capital of France?", "response": "Paris"},
    {"instruction": "What is the largest planet?", "response": "Jupiter"},
    {"instruction": "Who wrote Romeo and Juliet?", "response": "Shakespeare"},
    # Add more examples (at least 20 for meaningful fine‑tuning)
]

def format_example(example):
    return f"Instruction: {example['instruction']}\nResponse: {example['response']}{tokenizer.eos_token}"

texts = [format_example(ex) for ex in data]
dataset = Dataset.from_dict({"text": texts})

# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [ ]:
# Apply LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["c_attn"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./lora_gpt2",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Uncomment to train
# trainer.train()

In [ ]:
# Generate after fine‑tuning
prompt = "Instruction: What is the capital of France?\nResponse:"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0]))